### In this notebook i am creating a intermediate level of DA project using the S&P 500 Stocks (daily updated) dataset

### Link: [S&P 500 Stocks (daily updated)](https://www.kaggle.com/datasets/andrewmvd/sp-500-stocks)

In [2]:
import pandas as pd
import os
print(os.listdir("../Data"))

['sp500_stocks.csv', 'sp500_companies.csv', 'sp500_index.csv']


In [5]:
#Loading the Data
df = pd.read_csv('../Data/sp500_stocks.csv')

print(df.head())
print(df.info())


         Date Symbol  Adj Close  Close  High  Low  Open  Volume
0  2010-01-04    MMM        NaN    NaN   NaN  NaN   NaN     NaN
1  2010-01-05    MMM        NaN    NaN   NaN  NaN   NaN     NaN
2  2010-01-06    MMM        NaN    NaN   NaN  NaN   NaN     NaN
3  2010-01-07    MMM        NaN    NaN   NaN  NaN   NaN     NaN
4  2010-01-08    MMM        NaN    NaN   NaN  NaN   NaN     NaN
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1891536 entries, 0 to 1891535
Data columns (total 8 columns):
 #   Column     Dtype  
---  ------     -----  
 0   Date       object 
 1   Symbol     object 
 2   Adj Close  float64
 3   Close      float64
 4   High       float64
 5   Low        float64
 6   Open       float64
 7   Volume     float64
dtypes: float64(6), object(2)
memory usage: 115.5+ MB
None


### Data cleaning and Wragling

In [6]:
#Convert Date

df["Date"] = pd.to_datetime(df["Date"])

In [7]:
#Check Missing Values
print(df.isnull().sum())

Date               0
Symbol             0
Adj Close    1273705
Close        1273705
High         1273705
Low          1273705
Open         1273705
Volume       1273705
dtype: int64


### Removing the rows with missing prices

In [9]:
df= df.dropna(
    subset=["Open", "High", "Low", "Close"]
)

df

,Date,Symbol,Adj Close,Close,High,Low,Open,Volume
3768,2010-01-04,AOS,5.937266,7.435000,7.480000,7.261667,7.295000,1104600.0
3769,2010-01-05,AOS,5.861404,7.340000,7.431667,7.308333,7.431667,1207200.0
3770,2010-01-06,AOS,5.864068,7.343333,7.405000,7.301667,7.335000,663000.0
3771,2010-01-07,AOS,5.881369,7.365000,7.425000,7.311667,7.356667,564000.0
3772,2010-01-08,AOS,5.967879,7.473333,7.485000,7.311667,7.331667,504600.0
...,...,...,...,...,...,...,...,...
1876459,2024-12-16,XYL,120.779999,120.779999,122.570000,120.000000,120.720001,1515900.0
1876460,2024-12-17,XYL,120.769997,120.769997,121.760002,119.730003,119.730003,2009200.0
1876461,2024-12-18,XYL,116.919998,116.919998,121.559998,116.879997,120.790001,1638500.0
1876462,2024-12-19,XYL,116.430000,116.430000,118.919998,116.129997,117.440002,1708000.0


In [11]:
#Fill the misssing volumes

df["Volume"] = df["Volume"].fillna(0)

df

,Date,Symbol,Adj Close,Close,High,Low,Open,Volume
3768,2010-01-04,AOS,5.937266,7.435000,7.480000,7.261667,7.295000,1104600.0
3769,2010-01-05,AOS,5.861404,7.340000,7.431667,7.308333,7.431667,1207200.0
3770,2010-01-06,AOS,5.864068,7.343333,7.405000,7.301667,7.335000,663000.0
3771,2010-01-07,AOS,5.881369,7.365000,7.425000,7.311667,7.356667,564000.0
3772,2010-01-08,AOS,5.967879,7.473333,7.485000,7.311667,7.331667,504600.0
...,...,...,...,...,...,...,...,...
1876459,2024-12-16,XYL,120.779999,120.779999,122.570000,120.000000,120.720001,1515900.0
1876460,2024-12-17,XYL,120.769997,120.769997,121.760002,119.730003,119.730003,2009200.0
1876461,2024-12-18,XYL,116.919998,116.919998,121.559998,116.879997,120.790001,1638500.0
1876462,2024-12-19,XYL,116.430000,116.430000,118.919998,116.129997,117.440002,1708000.0


### Ensuring the correct datatypes

In [16]:
price_cols = ["Open", "High", "Low", "Close"]

df[price_cols] = df[price_cols].astype(float)

df["Volume"] = df["Volume"].astype(int)

df



,Date,Symbol,Adj Close,Close,High,Low,Open,Volume
3768,2010-01-04,AOS,5.937266,7.435000,7.480000,7.261667,7.295000,1104600
3769,2010-01-05,AOS,5.861404,7.340000,7.431667,7.308333,7.431667,1207200
3770,2010-01-06,AOS,5.864068,7.343333,7.405000,7.301667,7.335000,663000
3771,2010-01-07,AOS,5.881369,7.365000,7.425000,7.311667,7.356667,564000
3772,2010-01-08,AOS,5.967879,7.473333,7.485000,7.311667,7.331667,504600
...,...,...,...,...,...,...,...,...
1876459,2024-12-16,XYL,120.779999,120.779999,122.570000,120.000000,120.720001,1515900
1876460,2024-12-17,XYL,120.769997,120.769997,121.760002,119.730003,119.730003,2009200
1876461,2024-12-18,XYL,116.919998,116.919998,121.559998,116.879997,120.790001,1638500
1876462,2024-12-19,XYL,116.430000,116.430000,118.919998,116.129997,117.440002,1708000


### Filter Delisted Stocks

In [17]:
latest_date = df["Date"].max()

active_stocks = (
    df.groupby("Symbol")["Date"].max()
)

active_stocks = active_stocks[active_stocks > latest_date - pd.Timedelta(days=30)].index

df = df[df["Symbol"].isin(active_stocks)]

df

,Date,Symbol,Adj Close,Close,High,Low,Open,Volume
3768,2010-01-04,AOS,5.937266,7.435000,7.480000,7.261667,7.295000,1104600
3769,2010-01-05,AOS,5.861404,7.340000,7.431667,7.308333,7.431667,1207200
3770,2010-01-06,AOS,5.864068,7.343333,7.405000,7.301667,7.335000,663000
3771,2010-01-07,AOS,5.881369,7.365000,7.425000,7.311667,7.356667,564000
3772,2010-01-08,AOS,5.967879,7.473333,7.485000,7.311667,7.331667,504600
...,...,...,...,...,...,...,...,...
1876459,2024-12-16,XYL,120.779999,120.779999,122.570000,120.000000,120.720001,1515900
1876460,2024-12-17,XYL,120.769997,120.769997,121.760002,119.730003,119.730003,2009200
1876461,2024-12-18,XYL,116.919998,116.919998,121.559998,116.879997,120.790001,1638500
1876462,2024-12-19,XYL,116.430000,116.430000,118.919998,116.129997,117.440002,1708000


### Feature Engineering

In [18]:
df = df.sort_values(
    ["Symbol", "Date"]
)

### Daily Return Formula:

$$
\frac{P_t-P_{t-1}}{P_{t-1}}
$$

In [21]:
df["Daily_Return"] = (
    df.groupby("Symbol")["Close"].pct_change()
)

### Percentage Change

In [22]:
df["pct_change"] = (
    (df["Close"] - df["Open"])
    / df["Open"]
)* 100

### 7-Day Moving Average

In [26]:
df["MA7"] = (
    df.groupby("Symbol")["Close"]
    .transform(
        lambda x: x.rolling(7).mean()
    )
)

### 30-Day Moving Average

In [27]:
df["MA30"] = (
    df.groupby("Symbol")["Close"]
    .transform(
        lambda x: x.rolling(30).mean()
    )
)

### Volatality

In [28]:
df["Volatality"] =(
    df.groupby("Symbol")["Daily_Return"]
    .transform(
        lambda x: x.rolling(30).std()
    )
)

### Volume Change

In [29]:
df["Volume_Change"] = (
    df.groupby("Symbol")["Volume"]
    .pct_change()
)